# 大语言模型低比特量化

> 上一章分析了大语言模型的推理过程：在 Decode 阶段，每生成一个 Token，都要重新读取大量模型权重。一个 7B 模型使用 BF16 权重时，仅参数就约占 14 GB；如果机器只有 8 GB 显存，模型甚至无法完整加载。
>
> 低比特量化用更少的位数表示模型中的数字，让大模型占用更少的存储、显存与内存带宽。但 Hugging Face 上同时存在 `W4A16`、`W8A8`、`GPTQ`、`AWQ`、`FP8`、`Q3_K_M` 等名称：它们描述的并不是同一件事。
>
> 本章围绕“如何选择并运行一个低比特大模型”介绍四组核心内容：
>
> 1. **低比特表示**：通过 scale、zero point 与量化粒度，把连续的浮点数映射到有限的低比特位置。
>
> 2. **权重量化**：理解 `W8A16`、`W4A16`，以及 RTN、GPTQ、AWQ 如何在压缩权重的同时控制误差。
>
> 3. **激活与 KV Cache 量化**：理解 `W8A8`、FP8、Activation Outlier、SmoothQuant、PTQ、QAT 与 KV Cache 量化。
>
> 4. **模型格式与运行**：看懂 Hugging Face 中的量化配置和 GGUF 名称，下载模型，并用 Transformers、vLLM 或 llama.cpp 运行。

先从最直接的问题开始：不同精度下，一个 7B 模型的权重究竟有多大？


In [ ]:
def model_weight_size_gb(params_billion, bits_per_weight):
    """只估算模型权重，不包含 KV Cache、activation 和运行时开销。"""
    return params_billion * 1e9 * bits_per_weight / 8 / 1e9


params_billion = 7
formats = {
    "FP32": 32,
    "FP16 / BF16": 16,
    "INT8": 8,
    "INT4 (ideal)": 4,
}

print(f"{params_billion}B 模型的理论权重大小：")
for name, bits in formats.items():
    size = model_weight_size_gb(params_billion, bits)
    print(f"  {name:<14} {size:>5.1f} GB")

print("\nINT4 的理论权重大小只有 BF16 的四分之一。")
print("实际文件还会保存 scale、元数据和部分高精度张量，因此通常略大。")


## 1. 从浮点数到低比特

BF16 或 FP16 使用 16 bit 表示一个权重，而 INT4 只有 4 bit，只能表示 16 个离散位置。量化要做的事情，就是用这 16 个位置近似一组连续浮点数。

```text
浮点权重 → 确定数值范围 → 计算 scale → 映射到低比特整数 → 保存
                                                        ↓
模型计算 ← 低精度 Kernel 或反量化 ← 读取低比特整数与 scale
```

先看最简单的对称 INT4。它通常使用 `[-7, 7]` 这 15 个整数位置，并让浮点数 `0` 精确映射到整数 `0`：

$$
s = \frac{\max |x|}{7}, \qquad
q = \operatorname{clip}\left(\operatorname{round}\left(\frac{x}{s}\right), -7, 7\right)
$$

反量化时再乘回 scale：

$$
\hat{x} = s q
$$

量化后的 $q$ 是真正保存的低比特整数，$\hat{x}$ 是它近似还原出的浮点值。


In [ ]:
import numpy as np


def symmetric_quantize_dequantize(x, n_bits=4, axis=None):
    """对称量化后立即反量化，便于观察误差。"""
    qmax = 2 ** (n_bits - 1) - 1
    max_abs = np.max(np.abs(x), axis=axis, keepdims=True)
    scale = np.maximum(max_abs / qmax, 1e-12)
    q = np.round(x / scale).clip(-qmax, qmax).astype(np.int32)
    x_hat = q.astype(np.float32) * scale
    return q, x_hat, scale


x = np.array([-1.0, -0.72, -0.31, 0.0, 0.18, 0.63, 1.0], dtype=np.float32)
q, x_hat, scale = symmetric_quantize_dequantize(x, n_bits=4)

print(f"scale = {float(scale.squeeze()):.4f}")
print("原始浮点数: ", np.round(x, 3))
print("INT4 整数:  ", q)
print("反量化结果: ", np.round(x_hat, 3))
print(f"MAE = {np.mean(np.abs(x - x_hat)):.4f}")


上面的每个整数位置都代表一个宽度为 `scale` 的区间。scale 越小，能分辨的差异越细；但 scale 太小又会覆盖不了较大的值，造成 clipping。量化的核心取舍，就是用有限的位置覆盖多大的范围。

### Scale、Zero Point 与量化粒度

对称量化适合大致以 0 为中心的权重。如果数据明显偏向一侧，例如 ReLU 后全部非负的 Activation，可以使用非对称量化，把整数区间整体平移。

设整数范围为 $[q_{\min}, q_{\max}]$，标准 affine quantization 为：

$$
s = \frac{x_{\max} - x_{\min}}{q_{\max} - q_{\min}}
$$

$$
z = \operatorname{clip}\left(\operatorname{round}\left(q_{\min} - \frac{x_{\min}}{s}\right), q_{\min}, q_{\max}\right)
$$

$$
q = \operatorname{clip}\left(\operatorname{round}\left(\frac{x}{s}\right) + z, q_{\min}, q_{\max}\right),
\qquad \hat{x} = s(q-z)
$$

这里的 zero point $z$ 是**整数域中的偏移量**，不是浮点数据的最小值。它保证真实的 0 可以映射到一个整数位置。


In [ ]:
def affine_quantize_dequantize(x, n_bits=4):
    """非对称 uint 量化：整数范围为 [0, 2^n_bits - 1]。"""
    qmin, qmax = 0, 2**n_bits - 1
    x_min, x_max = float(np.min(x)), float(np.max(x))
    scale = max((x_max - x_min) / (qmax - qmin), 1e-12)
    zero_point = int(np.clip(np.round(qmin - x_min / scale), qmin, qmax))
    q = np.clip(np.round(x / scale) + zero_point, qmin, qmax).astype(np.int32)
    x_hat = scale * (q.astype(np.float32) - zero_point)
    return q, x_hat, scale, zero_point


activation = np.array([-0.40, 0.0, 0.12, 0.45, 0.92, 1.80, 3.00], dtype=np.float32)
q, activation_hat, scale, zero_point = affine_quantize_dequantize(activation)

print(f"scale = {scale:.4f}, zero_point = {zero_point}")
print("原始 Activation:", activation)
print("UINT4 整数:     ", q)
print("反量化结果:     ", np.round(activation_hat, 3))
print(f"MAE = {np.mean(np.abs(activation - activation_hat)):.4f}")


上面一整段数据共用一个 scale，这叫 **per-tensor** 量化。真实 LLM 的权重矩阵很大，不同区域的数值范围可能相差很多。如果所有权重共享一个 scale，少量大值会让 scale 变粗，其他小权重只能挤在少数整数位置中。

常见量化粒度如下：

| 粒度 | Scale 的共享范围 | 特点 |
|:---|:---|:---|
| Per-tensor | 整个张量 | 参数少、实现简单，容易受 Outlier 影响 |
| Per-channel | 每个输出通道 | 权重常用，能适应不同通道的范围 |
| Per-group | 每组权重 | 4-bit LLM 常用，在精度与 scale 开销之间折中 |
| Per-token | 每个 Token 的 Activation | 适应输入变化，常用于动态 Activation 量化 |

下面构造一个不同通道范围相差很大的权重矩阵，比较三种粒度。


In [ ]:
import matplotlib.pyplot as plt


np.random.seed(7)
W = np.random.randn(4, 16).astype(np.float32) * 0.25
W[1] *= 8.0  # 模拟一个范围明显更大的输出通道


def groupwise_quantize_dequantize(W, n_bits=4, group_size=4):
    assert W.shape[1] % group_size == 0
    restored = np.empty_like(W)
    for row in range(W.shape[0]):
        for start in range(0, W.shape[1], group_size):
            block = W[row, start : start + group_size]
            _, block_hat, _ = symmetric_quantize_dequantize(block, n_bits=n_bits)
            restored[row, start : start + group_size] = block_hat
    return restored


_, W_tensor, _ = symmetric_quantize_dequantize(W, n_bits=4)
_, W_channel, _ = symmetric_quantize_dequantize(W, n_bits=4, axis=1)
W_group = groupwise_quantize_dequantize(W, n_bits=4, group_size=4)

errors = {
    "Per-tensor": np.mean(np.abs(W - W_tensor)),
    "Per-channel": np.mean(np.abs(W - W_channel)),
    "Per-group": np.mean(np.abs(W - W_group)),
}

for name, error in errors.items():
    print(f"{name:<12} INT4 MAE = {error:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
im = axes[0].imshow(W, cmap="RdBu_r", aspect="auto")
axes[0].set_title("Weight matrix with one outlier channel")
axes[0].set_xlabel("Input channel")
axes[0].set_ylabel("Output channel")
fig.colorbar(im, ax=axes[0], fraction=0.046)

axes[1].bar(errors.keys(), errors.values(), color=["#4C78A8", "#F58518", "#54A24B"])
axes[1].set_title("INT4 error by quantization granularity")
axes[1].set_ylabel("Mean absolute error")
axes[1].tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()


粒度越细，每个 scale 需要覆盖的范围越小，量化误差通常越低；代价是要保存更多 scale，Kernel 也要处理更复杂的分组。至此，我们已经知道低比特数字如何表示。接下来把它放回 LLM 的核心计算：

$$
Y = XW
$$

$W$ 是模型权重，$X$ 是输入 Activation。先只量化权重，就得到最常见的 Weight-only 路线。


## 2. 权重量化

权重量化把线性层、Embedding 等模型参数保存为更低精度，而 Activation 仍保持 FP16 或 BF16。常见写法为：

| 写法 | Weight | Activation | 常见含义 |
|:---|:---|:---|:---|
| W8A16 | INT8 / FP8 | FP16 / BF16 | 8-bit Weight-only |
| W4A16 | INT4 / FP4 | FP16 / BF16 | 4-bit Weight-only |

`W4A16` 只描述精度组合，并没有说明使用 GPTQ、AWQ 还是其他算法，也没有说明文件一定是 GGUF。

Weight-only 量化特别适合 Decode。Decode 每次只处理一个新 Token，计算量不大，却需要读取整层权重，通常受到显存带宽限制。把权重从 16 bit 压到 4 bit，可以显著减少权重读取量。高性能实现会在 Kernel 内完成读取、解包、反量化与矩阵乘法，而不是先把整个模型恢复成 FP16。

下面用一个 Linear Layer 模拟 W4A16：$W$ 量化成 4-bit，$X$ 保持 FP16。


In [ ]:
np.random.seed(21)
batch, in_features, out_features = 8, 64, 16
X = np.random.randn(batch, in_features).astype(np.float16)
W = (np.random.randn(in_features, out_features) * 0.15).astype(np.float32)


def quantize_weight_by_input_group(W, n_bits=4, group_size=16):
    """沿输入通道分组，模拟常见 W4A16 group-wise 权重量化。"""
    assert W.shape[0] % group_size == 0
    W_hat = np.empty_like(W)
    for start in range(0, W.shape[0], group_size):
        block = W[start : start + group_size, :]
        # 每个输出通道在当前 group 内使用一个 scale
        _, block_hat, _ = symmetric_quantize_dequantize(block, n_bits=n_bits, axis=0)
        W_hat[start : start + group_size, :] = block_hat
    return W_hat


W_int4_hat = quantize_weight_by_input_group(W, n_bits=4, group_size=16)
y_fp = X.astype(np.float32) @ W
y_w4a16 = X.astype(np.float32) @ W_int4_hat

relative_output_error = np.linalg.norm(y_fp - y_w4a16) / np.linalg.norm(y_fp)
print(f"Weight MAE: {np.mean(np.abs(W - W_int4_hat)):.5f}")
print(f"Relative output error: {relative_output_error:.4%}")
print("W4A16 数据流：FP16 Activation × 反量化后的 INT4 Weight")


### 从 RTN 到 GPTQ、AWQ

前面的实现使用 **RTN（Round-to-Nearest）**：按固定 scale 直接舍入到最近的整数。RTN 快且不需要数据，但它只关心单个权重的误差 $|W-\hat{W}|$，不知道这个误差会怎样影响模型输出。

真实 LLM 中，并不是所有权重同等重要。某个输入通道经常出现较大的 Activation，那么该通道上一点点权重误差，也可能明显改变输出。校准数据（Calibration Data）的作用，就是让量化算法先观察一小批真实输入，再判断哪些误差更危险。

假设 $Y=XW$，只改动第 $j$ 个输入通道的权重，输出误差会被 $X$ 的第 $j$ 列放大。下面用一个极小实验观察这种差异。


In [ ]:
np.random.seed(9)
X_calib = np.random.randn(256, 4).astype(np.float32)
X_calib[:, 2] *= 12.0  # 第 3 个输入通道在校准数据中非常活跃

W = np.array([[0.3], [-0.2], [0.4], [0.1]], dtype=np.float32)
same_weight_error = 0.05

importance = np.mean(X_calib**2, axis=0)  # Hessian 对角线的直观近似
print("Calibration importance E[x_j^2]:", np.round(importance, 2))
print()

for channel in range(4):
    W_perturbed = W.copy()
    W_perturbed[channel, 0] += same_weight_error
    output_mse = np.mean((X_calib @ W - X_calib @ W_perturbed) ** 2)
    print(f"通道 {channel}: 相同权重误差 {same_weight_error:.2f} → output MSE {output_mse:.5f}")

print("\n相同大小的权重误差，落在活跃通道上会造成更大的输出误差。")


这个实验解释了为什么现代 LLM 权重量化会使用校准数据：优化目标不只是让权重看起来接近，还要让量化前后的层输出接近。

| 方法 | 它解决的问题 | 核心思路 |
|:---|:---|:---|
| RTN | 快速得到一个低比特基线 | 按 scale 直接舍入，不使用校准数据 |
| GPTQ | 舍入一个权重后，误差会影响整层输出 | 使用近似二阶信息，按顺序量化并补偿尚未量化的权重 |
| AWQ | 少数重要通道对输出特别敏感 | 根据 Activation 统计识别重要权重通道，通过等价缩放降低误差 |

GPTQ 和 AWQ 都常用于 W4A16，但它们是**生成量化权重的方法**，不是数值格式。模型完成量化后，还需要对应的打包格式和推理 Kernel 才能真正节省显存并获得速度收益。

权重量化解决了模型“装不下”和 Decode 读取权重过多的问题。但在 $Y=XW$ 中，$X$ 仍然是高精度 Activation。要进一步利用低精度矩阵乘法，还需要考虑 Activation 量化。


## 3. 激活与 KV Cache 量化

Activation 是模型在处理输入时动态产生的中间结果。它和固定不变的 Weight 不同：输入内容、Token 位置和 Batch 都会改变 Activation 的分布。

| 写法 | Weight | Activation | 典型目标 |
|:---|:---|:---|:---|
| W8A8 | INT8 / FP8 | INT8 / FP8 | 使用低精度矩阵乘法提高吞吐 |
| W4A8 | INT4 / FP4 | INT8 / FP8 | 同时追求权重压缩与计算效率 |
| W4A4 | INT4 / FP4 | INT4 / FP4 | 更激进的低精度计算，需要专门硬件与算法 |

只看 `W8A8` 仍然不够：它可能是 INT8，也可能是 FP8；scale 可能是静态校准得到的，也可能在推理时按 Token 动态计算。最终能否加速取决于硬件和 Kernel 是否原生支持这条数据路径。

Activation 量化最典型的困难是 **Outlier Channel**：少数通道的数值比其他通道大很多。一个 per-tensor scale 为了容纳 Outlier，会让普通通道的量化间隔变得很粗。


In [ ]:
np.random.seed(11)
X_act = np.random.randn(64, 128).astype(np.float32) * 0.5
outlier_channels = [17, 63, 101]
X_act[:, outlier_channels] *= 40.0

_, X_int8_hat, scale = symmetric_quantize_dequantize(X_act, n_bits=8)
error = np.abs(X_act - X_int8_hat)

normal_mask = np.ones(X_act.shape[1], dtype=bool)
normal_mask[outlier_channels] = False

print(f"Activation range: [{X_act.min():.1f}, {X_act.max():.1f}]")
print(f"Per-tensor INT8 scale: {float(scale.squeeze()):.4f}")
print(f"Normal-channel MAE: {error[:, normal_mask].mean():.4f}")
print(f"Outlier-channel MAE: {error[:, outlier_channels].mean():.4f}")
normal_relative_mae = error[:, normal_mask].mean() / np.abs(X_act[:, normal_mask]).mean()
outlier_relative_mae = error[:, outlier_channels].mean() / np.abs(X_act[:, outlier_channels]).mean()
print(f"Normal-channel relative MAE: {normal_relative_mae:.1%}")
print(f"Outlier-channel relative MAE: {outlier_relative_mae:.1%}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
channel_max = np.max(np.abs(X_act), axis=0)
axes[0].bar(np.arange(X_act.shape[1]), channel_max, color="#4C78A8", width=1.0)
axes[0].bar(outlier_channels, channel_max[outlier_channels], color="#E45756")
axes[0].set_title("Activation outlier channels")
axes[0].set_xlabel("Channel")
axes[0].set_ylabel("Maximum absolute value")

axes[1].hist(error[:, normal_mask].ravel(), bins=40, color="#F58518")
axes[1].set_title("Quantization error on normal channels")
axes[1].set_xlabel("Absolute error")
axes[1].set_ylabel("Count")
plt.tight_layout()
plt.show()


### SmoothQuant：把难度从 Activation 移到 Weight

SmoothQuant 观察到 Weight 通常比 Activation 更容易量化，因此通过等价变换，把 Activation 的 Outlier“平滑”一部分到 Weight。

对于 $Y=XW$，给每个输入通道选择一个正数 scale $s$：

$$
Y = XW = (X \operatorname{diag}(s)^{-1})(\operatorname{diag}(s)W)
$$

也就是让 Activation 除以 $s$，同时让对应的 Weight 乘以 $s$。浮点计算结果完全不变，但两边的数值范围变得更适合联合量化。SmoothQuant 正是沿着这条思路实现 W8A8 PTQ。


In [ ]:
np.random.seed(31)
X = np.random.randn(512, 16).astype(np.float32) * 0.5
X[:, [3, 11]] *= 25.0
W = np.random.randn(16, 8).astype(np.float32) * 0.2


def int8_tensor_qdq(x):
    return symmetric_quantize_dequantize(x, n_bits=8)[1]


y_ref = X @ W

# 直接 W8A8
y_naive = int8_tensor_qdq(X) @ int8_tensor_qdq(W)

# 教学版 SmoothQuant scale：平衡每个输入通道两侧的最大值
x_max = np.max(np.abs(X), axis=0)
w_max = np.max(np.abs(W), axis=1)
s = np.sqrt(np.maximum(x_max, 1e-8) / np.maximum(w_max, 1e-8))
X_smooth = X / s
W_smooth = W * s[:, None]

assert np.allclose(X @ W, X_smooth @ W_smooth, atol=1e-5)
y_smooth = int8_tensor_qdq(X_smooth) @ int8_tensor_qdq(W_smooth)

err_naive = np.linalg.norm(y_ref - y_naive) / np.linalg.norm(y_ref)
err_smooth = np.linalg.norm(y_ref - y_smooth) / np.linalg.norm(y_ref)

print(f"Original activation max: {np.max(np.abs(X)):.2f}")
print(f"Smoothed activation max: {np.max(np.abs(X_smooth)):.2f}")
print(f"Naive W8A8 relative error:      {err_naive:.4%}")
print(f"Smoothed W8A8 relative error:   {err_smooth:.4%}")


### PTQ 与 QAT

GPTQ、AWQ、SmoothQuant 通常属于 **PTQ（Post-Training Quantization）**：模型训练完成后，再用少量校准数据确定 scale、clipping 或权重调整，不需要重新进行完整训练。

**QAT（Quantization-Aware Training）** 则在训练或微调时插入 Fake Quantization：前向传播模拟低比特的舍入与 clipping，反向传播仍使用浮点梯度。模型会在训练过程中主动适应量化误差。

```text
PTQ: 浮点模型 → 校准 → 量化 → 评测 → 部署

QAT: 浮点模型 → 插入 Fake Quant → 继续训练 → Convert → 部署
```

PTQ 成本低，是大模型部署的常用起点；当位数更低、模型更敏感或 PTQ 无法恢复精度时，QAT 的价值更明显。下面用 Straight-Through Estimator 做一个极小的 Fake Quant 示例。


In [ ]:
np.random.seed(5)
train_x = np.random.randn(128, 4).astype(np.float32)
target_weight = np.random.randn(4, 2).astype(np.float32) * 0.3
target_y = train_x @ target_weight
weight = np.random.randn(4, 2).astype(np.float32) * 0.3


def fake_quant_int4(weight):
    """前向传播使用 INT4 量化-反量化后的权重。"""
    return symmetric_quantize_dequantize(weight, n_bits=4)[1]


learning_rate = 0.08
initial_loss = np.mean((train_x @ fake_quant_int4(weight) - target_y) ** 2)
for _ in range(160):
    # Forward：模型实际看见 Fake Quant 后的权重
    quant_weight = fake_quant_int4(weight)
    prediction = train_x @ quant_weight
    error = prediction - target_y

    # Backward：STE 把 round 近似成恒等函数，梯度更新浮点 master weight
    grad_weight = (2.0 / train_x.shape[0]) * train_x.T @ error
    weight -= learning_rate * grad_weight

final_quant_weight = fake_quant_int4(weight)
final_loss = np.mean((train_x @ final_quant_weight - target_y) ** 2)
print(f"QAT toy example initial loss: {initial_loss:.6f}")
print(f"QAT toy example final loss: {final_loss:.6f}")
print("前向传播看见量化误差，反向传播更新浮点 master weight。")


### KV Cache 量化

Weight 和 Activation 之外，自回归推理还会保存每一层历史 Token 的 Key 与 Value。KV Cache 大小随 Batch、上下文长度和层数线性增长：

$$
\text{KV bytes} = B \times L \times T \times H_{kv} \times D \times 2 \times \text{bytes}
$$

其中最后的 `2` 表示 Key 和 Value。长上下文或高并发时，KV Cache 可能成为主要显存开销。将 BF16 KV Cache 改为 FP8，可以把这部分理论内存减半；但同样需要合适的校准 scale 与 Attention Kernel。


In [ ]:
def kv_cache_gb(batch, layers, tokens, kv_heads, head_dim, bits):
    elements = batch * layers * tokens * kv_heads * head_dim * 2  # K + V
    return elements * bits / 8 / 1024**3


config = dict(batch=8, layers=32, kv_heads=8, head_dim=128)
print("Llama-style GQA config, batch=8")
print(f"{'Context':>10} {'BF16 KV':>12} {'FP8 KV':>12}")
print("-" * 38)
for tokens in [4096, 32768, 131072]:
    bf16 = kv_cache_gb(tokens=tokens, bits=16, **config)
    fp8 = kv_cache_gb(tokens=tokens, bits=8, **config)
    print(f"{tokens:>10,} {bf16:>9.2f} GB {fp8:>9.2f} GB")


现在可以把大语言模型中的三类量化对象放在一起：

```text
Weight      → 模型大小与 Decode 权重带宽 → W4A16、GPTQ、AWQ、GGUF
Activation  → 低精度矩阵乘法与吞吐     → W8A8、FP8、SmoothQuant、QAT
KV Cache    → 长上下文与并发显存         → FP8 KV、INT8 KV
```

理解这些层级之后，再看 Hugging Face 上的一串名称，就可以逐层拆解，而不需要把每个名称当成一种全新的技术。


## 4. 量化模型的格式与运行

Hugging Face 上常见的模型名称可能是：

```text
Model-BF16
Model-GPTQ-Int4-128g
Model-AWQ
Model-bnb-4bit
Model-FP8
Model-GGUF
```

它们可以按下面的顺序阅读：

| 名称片段 | 描述的层级 | 需要继续确认什么 |
|:---|:---|:---|
| W4A16、W8A8 | Weight 与 Activation 精度组合 | INT 还是 FP、静态还是动态 |
| GPTQ、AWQ、SmoothQuant | 量化或误差校正方法 | bits、group size、zero point |
| INT4、FP8、NF4 | 数值表示 | 粒度、Kernel 与硬件支持 |
| SafeTensors、GGUF | 权重文件或模型容器 | 对应的加载方式与 Runtime |
| Marlin、CUTLASS、ggml | 执行低比特计算的 Kernel | GPU/CPU 是否支持 |
| Transformers、vLLM、llama.cpp | 加载与推理系统 | 是否支持该量化配置 |

例如 `GPTQ-Int4-128g` 通常表示：使用 GPTQ 生成 INT4 权重，group size 为 128；Activation 多数仍保持 FP16/BF16。真正加载时仍应查看模型仓库中的 `quantization_config`，不能只依赖名称猜测。

### SafeTensors 与 Hugging Face 量化配置

GPU 生态的 AWQ、GPTQ、FP8 模型通常以 SafeTensors 保存，并在 `config.json` 中记录 `quantization_config`。下载前重点检查：

```text
config.json                       模型结构与量化配置
model.safetensors                 单文件权重
model-00001-of-00004.safetensors  分片权重
model.safetensors.index.json      分片索引
tokenizer.json                    Tokenizer
*.gguf                            GGUF 模型文件
```

安装 Hugging Face CLI 后，可以下载整个仓库或单个文件：

```bash
pip install -U huggingface_hub

# 下载完整模型仓库
hf download <owner>/<model-repo> --local-dir ./models/model

# 只下载一个 GGUF 文件，避免把所有量化版本全部下载
hf download <owner>/<gguf-repo> <model>-Q4_K_M.gguf --local-dir ./models
```


### GGUF 与 Q3_K_M

GGUF 是 llama.cpp / ggml 生态使用的模型容器。它可以在一个或多个文件中保存权重、模型结构、Tokenizer 和其他元数据。GGUF 是文件格式，`Q3_K_M` 才是其中一种量化预设。

常见 GGUF 类型可以先分为三组：

| 家族 | 常见名称 | 特点 |
|:---|:---|:---|
| Legacy Quants | Q4_0、Q5_0、Q8_0 | 较早、兼容广，命名简单 |
| K-Quants | Q3_K_S/M/L、Q4_K_S/M、Q5_K_S/M、Q6_K | 以 block 分组并混合不同 Tensor 精度 |
| Importance Quants | IQ2_XXS、IQ3_M、IQ4_XS | 使用重要性信息改善极低比特质量 |

拆开 `Q3_K_M`：

```text
Q3  → 以 3-bit K-Quant 为核心
K   → K-Quant 家族
M   → Medium 混合预设
```

这里的名称不是整个模型严格的平均 bits/weight。不同 Tensor 可以使用不同精度，还要保存 scale、block metadata 等信息。以 llama.cpp 官方对 Llama 3.1 8B 的一组结果为例，`Q3_K_M` 实际约为 3.996 bpw，`Q4_K_M` 约为 4.894 bpw。


In [ ]:
gguf_bpw = {
    "Q3_K_S": 3.6429,
    "Q3_K_M": 3.9960,
    "Q3_K_L": 4.2979,
    "Q4_K_S": 4.6672,
    "Q4_K_M": 4.8944,
    "Q5_K_M": 5.7036,
    "Q6_K": 6.5633,
    "Q8_0": 8.5008,
}

params_billion = 7
print(f"按 llama.cpp 示例 bpw 粗略估算 {params_billion}B 权重大小：")
for name, bpw in gguf_bpw.items():
    size = model_weight_size_gb(params_billion, bpw)
    print(f"  {name:<8} {bpw:>6.3f} bpw  →  {size:>5.2f} GB")

print("\n名称中的数字是量化家族标签，不等于整个模型的实际平均 bpw。")


选择 GGUF 时，可以先从 `Q4_K_M` 作为质量与大小的平衡点，再根据机器内存和实际任务向两侧调整：内存更紧张时尝试 Q3/IQ 系列，质量更重要且内存充足时尝试 Q5、Q6 或 Q8。最终仍应使用自己的任务评测，而不是只看文件大小。

### 用 llama.cpp 运行 GGUF

llama.cpp 可以直接读取本地 GGUF，也可以从 Hugging Face 拉取兼容仓库：

```bash
# 本地文件
llama cli -m ./models/model-Q4_K_M.gguf -p "Explain low-bit quantization."

# 直接使用 Hugging Face GGUF 仓库
llama cli -hf <owner>/<model-GGUF>

# 启动 OpenAI-compatible API
llama serve -m ./models/model-Q4_K_M.gguf
```

也可以使用 Python 接口：

```python
from llama_cpp import Llama

llm = Llama(model_path="./models/model-Q4_K_M.gguf", n_ctx=4096)
result = llm("Low-bit quantization is", max_tokens=64)
print(result["choices"][0]["text"])
```

如果手里只有 Hugging Face BF16/FP16 模型，llama.cpp 的标准流程分为两步：先转成高精度 GGUF，再量化成目标类型。

```bash
python convert_hf_to_gguf.py ./model \
  --outfile model-BF16.gguf \
  --outtype bf16

llama-quantize model-BF16.gguf model-Q4_K_M.gguf Q4_K_M
```

不要从已经低比特量化过的模型再次量化。重复量化通常会累积误差，质量明显差于从 BF16/FP16 原始权重直接生成目标格式。


### 用 Transformers 与 vLLM 运行量化模型

Transformers 可以根据 Hugging Face 仓库中的 `quantization_config` 自动识别许多预量化模型：

```python
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "<owner>/<AWQ-or-GPTQ-model>"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    dtype="auto",
)
```

bitsandbytes 也可以在加载原始模型时进行 4-bit Weight-only 量化，NF4 常用于 QLoRA 等低显存微调场景：

```python
from transformers import BitsAndBytesConfig

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="bfloat16",
)
```

vLLM 更适合 GPU 服务。很多预量化仓库可以直接启动：

```bash
vllm serve <owner>/<quantized-model> --dtype auto
```

量化模型更小，不代表在任何硬件上都更快。选择方案时要同时确认：

| 场景 | 常见起点 | 原因 |
|:---|:---|:---|
| CPU / Apple Silicon 本地运行 | GGUF Q4_K_M + llama.cpp | 生态成熟，CPU/Metal Kernel 完整 |
| NVIDIA GPU 显存不足 | AWQ / GPTQ W4A16 | 显著减少权重显存与 Decode 带宽 |
| H100/更新 GPU 的高吞吐服务 | FP8 W8A8 | 可使用硬件原生低精度矩阵乘法 |
| 低显存 LoRA 微调 | bitsandbytes NF4 | 冻结低比特基座，只训练 LoRA 参数 |
| 长上下文或高并发 | FP8 / INT8 KV Cache | 减少随 Token 数增长的 Cache 显存 |

最终选择需要共同比较模型质量、显存、Prefill 吞吐、Decode tokens/s 和目标硬件支持。


### 看懂量化相关 JD

一段常见要求可能写成：

> 熟悉大模型 PTQ/QAT，了解 GPTQ、AWQ、SmoothQuant；具备 W4A16、W8A8、FP8 和 KV Cache 量化经验；熟悉 vLLM、TensorRT-LLM 或 llama.cpp，能够完成精度评测与性能分析。

现在可以按层级拆开：

- **PTQ/QAT**：量化发生在模型训练之后，还是训练过程中。
- **GPTQ/AWQ/SmoothQuant**：如何控制 Weight 或 Activation 的量化误差。
- **W4A16/W8A8/FP8**：Weight 与 Activation 使用什么精度组合和数值格式。
- **KV Cache 量化**：如何降低长上下文与高并发的显存占用。
- **vLLM/TensorRT-LLM/llama.cpp**：如何让量化模型在真实硬件上运行。
- **精度与性能分析**：不能只完成格式转换，还要证明质量损失和速度收益可以接受。

以后遇到新的量化名称，不需要立刻记住全部细节。先判断它在描述量化对象、数值格式、算法、模型文件，还是 Runtime，再去检查它需要的硬件与 Kernel。


## 小结

这一节所学的内容：

- 低比特量化通过 scale 和 zero point，把连续浮点数映射到有限的整数或浮点位置
- Per-tensor、Per-channel、Per-group、Per-token 描述 scale 的共享范围；粒度越细通常误差越低，但元数据和 Kernel 更复杂
- W4A16、W8A8 描述 Weight 与 Activation 的精度组合，不是具体量化算法
- 权重量化主要减少模型大小和 Decode 权重带宽；GPTQ 与 AWQ 常用于生成高质量 W4A16 权重
- Activation 随输入动态变化且存在 Outlier；SmoothQuant 通过等价变换把量化难度从 Activation 迁移到 Weight
- PTQ 在训练后使用校准数据完成量化；QAT 在训练时使用 Fake Quant 让模型适应误差
- KV Cache 量化主要服务长上下文与高并发，FP8 KV 可以把理论 Cache 内存减半
- GGUF 是模型容器，Q3_K_M、Q4_K_M 是 llama.cpp 的量化预设；名称中的数字不等于整个模型的实际平均 bpw
- SafeTensors/GGUF、GPTQ/AWQ、W4A16/W8A8、vLLM/llama.cpp 分属不同层级

面对一个真实量化模型，可以按下面的顺序判断：

```text
量化什么 → 使用什么精度 → 如何控制误差 → 保存成什么格式 → 用什么 Runtime 运行
```


## 作业

三道题分别检查低比特计算、模型名称解读和真实部署选择。可以让 AI 帮忙解释概念或检查命令，但请自己完成计算、判断和实际运行。


### 作业 1：手算 W4A16

给定一组权重：

```text
W = [-1.0, -0.5, 0.0, 0.4, 1.0]
```

使用对称 INT4 量化，手算 scale、量化整数和反量化结果。Activation 保持 FP16，因此这个组合记作 W4A16。


In [ ]:
import numpy as np

W = np.array([-1.0, -0.5, 0.0, 0.4, 1.0], dtype=np.float32)

# TODO：对称 INT4 的 qmax
qmax = None

# TODO：计算 scale
scale = None

# TODO：计算量化整数与反量化权重
W_q = None
W_hat = None

assert qmax == 7, "对称 INT4 的正整数上限应为 7"
assert scale is not None
assert W_q is not None
assert W_hat is not None
assert np.all(W_q >= -7) and np.all(W_q <= 7)

print("INT4 integers:", W_q)
print("Dequantized weights:", np.round(W_hat, 4))
print("作业 1 通过：你完成了一次 W4A16 权重量化。")


### 作业 2：解读量化模型名称

解释下面三个名称分别告诉了你什么、没有告诉你什么：

```text
Model-GPTQ-Int4-128g
Model-FP8
Model-Q3_K_M.gguf
```

至少回答：量化对象、数值格式或算法、文件格式、可能使用的 Runtime，以及还需要打开哪个配置文件继续确认。


### 作业 3：下载并运行一个 GGUF 模型

选择一个参数量较小的开源 Instruct 模型，在 Hugging Face 找到它的 GGUF 仓库：

1. 根据机器内存选择 Q3_K_M、Q4_K_M 或 Q5_K_M。
2. 使用 `hf download` 只下载选中的 GGUF 文件。
3. 使用 llama.cpp 或 llama-cpp-python 生成至少 50 个 Token。
4. 记录文件大小、加载内存、生成速度和一条输出。
5. 解释为什么选择这个量化类型；如果换成更低或更高精度，预期会发生什么。


## 参考资料

- Frantar et al., [GPTQ: Accurate Post-Training Quantization for Generative Pre-trained Transformers](https://arxiv.org/abs/2210.17323), 2022
- Lin et al., [AWQ: Activation-aware Weight Quantization for LLM Compression and Acceleration](https://arxiv.org/abs/2306.00978), 2023
- Xiao et al., [SmoothQuant: Accurate and Efficient Post-Training Quantization for Large Language Models](https://arxiv.org/abs/2211.10438), 2022
- Dettmers et al., [LLM.int8(): 8-bit Matrix Multiplication for Transformers at Scale](https://arxiv.org/abs/2208.07339), 2022
- Dettmers et al., [QLoRA: Efficient Finetuning of Quantized LLMs](https://arxiv.org/abs/2305.14314), 2023
- Hugging Face, [Transformers Quantization](https://huggingface.co/docs/transformers/quantization/overview)
- vLLM, [Quantization](https://docs.vllm.ai/en/latest/features/quantization/)
- ggml-org, [llama.cpp Quantization](https://github.com/ggml-org/llama.cpp/blob/master/tools/quantize/README.md)
